In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.quantum_info import Pauli
from qiskit_algorithms import QAOA
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import COBYLA

In [3]:
H1 = SparsePauliOp('ZZ')
sampler = StatevectorSampler()

qaoa = QAOA(
    sampler=sampler,
    optimizer=COBYLA(),
    reps=1
)

result = qaoa.compute_minimum_eigenvalue(H1)

print("optimal value:", result.optimal_value)
print("optimal point:", result.optimal_point)
print("optimal parameters:", result.optimal_parameters)
print("eigenstate:", result.eigenstate)

/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:603: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


optimal value: -0.99609375
optimal point: [5.09511523 2.371417  ]
optimal parameters: {ParameterVectorElement(β[0]): np.float64(5.095115233073935), ParameterVectorElement(γ[0]): np.float64(2.371416998502403)}
eigenstate: {'01': 0.501953125, '10': 0.498046875}


Definimos ahora un problema QUBO al que aplicarle el algoritmo QAOA

In [1]:
from qiskit_optimization.problems import QuadraticProgram

In [ ]:
# se crea un objeto que representa un problema 
# de optimización cuadrática (QUBO / QP)
# mediante la creacion deuna instancia (objeto) de la clase QuadraticProgram
qp = QuadraticProgram()

# se llama a un método del objeto qp que modifica su estado interno 
# añadiendo una variable
qp.binary_var('x')
qp.binary_var('y')
qp.binary_var('z')

# se llama a un método que establece atributos internos 
# del objeto relacionados con la función objetivo
qp.minimize(linear={'y':-1}, quadratic={('x','y'):2, ('z','y'):-4})

# se llama a un método que añade una restricción al estado interno del objeto qp
qp.linear_constraint(linear={'x':1, 'y':2, 'z':3}, sense='<=', rhs=5)

# se imprime el problema 
# (en representación estándar de optimización)
print(qp.prettyprint())

Problem name: 

Minimize
  2*x*y - 4*y*z - y

Subject to
  Linear constraints (1)
    x + 2*y + 3*z <= 5  'c0'

  Binary variables (3)
    x y z



Una vez tenemos un objeto QuadraticProgram, podemos resolverlo con uno de los algoritmos que inlcuye qiskit. Para ello usamos MinimumEigenOptimizer junto con un solver. 

EL OBJETO TODAVIA NO ESTA EN QUBO PORQUE TIENE RESTRICCIONES -> LOS ALGORITMOS USARAN VARIABLES DE SLACK PARA TRANSFORMARLOS A QUBO

- Podemos emplear un algoritmo clasico que prueba todas las soluciones posibles y selecciona la optima

In [11]:
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver

In [ ]:
# se crea una instancia de la clase NumPyMinimumEigensolver
# es un eigensolver clasico exacto que encuentra el autovalor minimo de un operador
# mediante diagonalizacion de matrices -> clasico
np_solver = NumPyMinimumEigensolver()

# se crea una instancia de la clase MinimumEigenOptimizer
# aun no se ha recibido el problema (qp)
# queda configurado para que cuando se llame a solver(qp) 
# transforme un problema de optimización (QuadraticProgram / QUBO)
# en un problema de autovalores (Hamiltoniano tipo Ising)
# delega la resolución al eigensolver (np_solver)
np_optimizer = MinimumEigenOptimizer(np_solver)

# se resuelve el problema
# 1. Convierte qp → QUBO (si hace falta)
# 2. QUBO → Hamiltoniano (operador)
# 3. Llama al eigensolver para encontrar el mínimo autovalor
# 4. Toma los resultados y da la solución del problema original
result = np_optimizer.solve(qp)

# Contiene:
# 1. valor óptimo
# 2. variables óptimas
# 3. estado de la solución
print(result)

fval=-5.0, x=0.0, y=1.0, z=1.0, status=SUCCESS


- Tambien podemos emplear QAOA para resolver el problema

In [ ]:
from qiskit_aer import Aer
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

In [ ]:
# se instancia el sampleador de vectores de estado
# para evaluar el circuito cuantico y obtener distribuciones y estados
sampler = StatevectorSampler()

# se instancia QAOA, algoritmo hibrido que busca aproximar el 
# estado de minima energia de un Hamiltoniano
qaoa = QAOA(
    sampler=sampler,
    optimizer=COBYLA(),
    reps=1
)

# se instancia MinimumEigenOptimizer
# Queda configurado para cuando se llame a solve(qp),
# transforme el problema de optimizacion en un problema de 
# autovalor minimo y delegue la resolucion a QAOA
qaoa_optimizer = MinimumEigenOptimizer(qaoa)

# Recibe qp, lo convierte en formulacion adecuada: 
# Quadratic Programming -> QUBO -> Hamiltoniano tipo Ising
# llama a qaoa, reconstruye la solucion
result = qaoa_optimizer.solve(qp)

# Contiene:
# 1. valor óptimo
# 2. variables óptimas
# 3. estado de la solución
print(result)

/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:603: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/_index.py:108: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


fval=-5.0, x=0.0, y=1.0, z=1.0, status=SUCCESS


In [ ]:
# result es el resultado del metodo solve propio de la clase 
# MinimumEigenOptimizer -> 
# result es un objeto de la clase MinimumEigenOptimizationResult
# con atributos, entre otros, variables y min_eigen_solver_result
# variables: Returns the list of variables of the optimization problem.
# min_eigen_solver_result: Returns a result object obtained from the instance of
# SamplingMinimumEigensolver or NumPyMinimumEigensolver.

print(f'Variable order: {(var.name for var in result.variables)}')
for s in result.samples:
    print(s)

Variable order: <generator object <genexpr> at 0x119c4dc00>
SolutionSample(x=array([0., 1., 1.]), fval=np.float64(-5.0), probability=0.0049285888671875, status=<OptimizationResultStatus.SUCCESS: 0>)
SolutionSample(x=array([0., 1., 0.]), fval=np.float64(-1.0), probability=0.002399444580078125, status=<OptimizationResultStatus.SUCCESS: 0>)
SolutionSample(x=array([1., 0., 1.]), fval=np.float64(0.0), probability=0.0042724609375, status=<OptimizationResultStatus.SUCCESS: 0>)
SolutionSample(x=array([0., 0., 0.]), fval=np.float64(0.0), probability=0.0014448165893554688, status=<OptimizationResultStatus.SUCCESS: 0>)
SolutionSample(x=array([0., 0., 1.]), fval=np.float64(0.0), probability=0.006464958190917969, status=<OptimizationResultStatus.SUCCESS: 0>)
SolutionSample(x=array([1., 0., 0.]), fval=np.float64(0.0), probability=0.00466156005859375, status=<OptimizationResultStatus.SUCCESS: 0>)
SolutionSample(x=array([1., 1., 0.]), fval=np.float64(1.0), probability=0.005633354187011719, status=<Opt

Hemos impreso las soluciones que son parte de la solucion optima que ha encontrado QAOA. 

In [22]:
print(result.min_eigen_solver_result)

{   'aux_operators_evaluated': None,
    'best_measurement': {   'bitstring': '000110',
                            'probability': 0.0556640625,
                            'state': 6,
                            'value': np.complex128(-52+0j)},
    'cost_function_evals': 34,
    'eigenstate': {   '000000': 0.01171875,
                      '000001': 0.03125,
                      '000010': 0.0107421875,
                      '000011': 0.01171875,
                      '000100': 0.021484375,
                      '000101': 0.0380859375,
                      '000110': 0.0087890625,
                      '000111': 0.001953125,
                      '001000': 0.0029296875,
                      '001010': 0.00390625,
                      '001011': 0.05859375,
                      '001100': 0.072265625,
                      '001101': 0.02734375,
                      '001110': 0.0048828125,
                      '001111': 0.0322265625,
                      '010000': 0.0029296875,
     

Podemos obtener el problema QUBO asociado al problema original

In [23]:
from qiskit_optimization.converters import QuadraticProgramToQubo
qp_to_qubo = QuadraticProgramToQubo()
qubo = qp_to_qubo.convert(qp)
print(qubo.prettyprint())

Problem name: 

Minimize
  8*c0@int_slack@0^2 + 32*c0@int_slack@0*c0@int_slack@1
  + 32*c0@int_slack@0*c0@int_slack@2 + 32*c0@int_slack@1^2
  + 64*c0@int_slack@1*c0@int_slack@2 + 32*c0@int_slack@2^2 + 16*x*c0@int_slack@0
  + 32*x*c0@int_slack@1 + 32*x*c0@int_slack@2 + 8*x^2 + 34*x*y + 48*x*z
  + 32*y*c0@int_slack@0 + 64*y*c0@int_slack@1 + 64*y*c0@int_slack@2 + 32*y^2
  + 92*y*z + 48*z*c0@int_slack@0 + 96*z*c0@int_slack@1 + 96*z*c0@int_slack@2
  + 72*z^2 - 80*c0@int_slack@0 - 160*c0@int_slack@1 - 160*c0@int_slack@2 - 80*x
  - 161*y - 240*z + 200

Subject to
  No constraints

  Binary variables (6)
    x y z c0@int_slack@0 c0@int_slack@1 c0@int_slack@2



y ahi se pueden ver las variables de slack

En qiskit_optimization.converters tambien estan InequalityToEquality, IntegerToBinary, LinearEqualityToPenalty -> procesos necesarios para pasar de Quadratic program a QUBO

QuadraticToQubo usa todas las anteriores para dar el problema en formulacion QUBO